# 1. Consolidação da base de dados (Censo 2022)

Este notebook consolida as tabelas de **Características dos Domicílios** dos Agregados por Setores Censitários do Censo Demográfico 2022 (IBGE), já agregadas por município, para o Estado de São Paulo.

- **Entrada:** arquivos em `1_DATA/raw/`
- **Saídas:** `base_final_SP.csv` (características dos domicílios) e `favelas_suplementar_SP.csv` (população em favelas e comunidades urbanas, usada apenas como variável suplementar, fora do agrupamento)

Após a execução, os dois arquivos devem ser colocados em `1_DATA/processed/`. O recorte da RMSP é aplicado no notebook de análise.

In [ ]:
# 1. DATA PROCESSING AND BASE CONSOLIDATION
# Only household characteristics (Censo 2022 - Agregados por Setores Censitarios).
# The favela population (N_POP_DPPO) comes from a separate IBGE dataset and is NOT
# part of the main base. It is saved in a separate file to be used later as a
# supplementary variable (overlaid on the clusters, outside the K-Means).

import pandas as pd

# Raw input files (IBGE) stored in 1_DATA/raw of this repository
path = "https://raw.githubusercontent.com/julianevesa/rmsp-habitacao-censo2022/main/1_DATA/raw/"

# Data Loading (household characteristics only)
dom1 = pd.read_csv(path + "ACM_SP.csv", encoding="latin1", sep=";").drop(
    columns=["NM_MUN", "NM_UF", "CD_UF"], errors="ignore"
)
dom2 = pd.read_csv(path + "ACM_SP2.csv", encoding="latin1", sep=";").drop(
    columns=["NM_MUN", "NM_UF", "CD_UF"], errors="ignore"
)
dom3 = pd.read_csv(path + "ACM_SP3.csv", encoding="latin1", sep=";").drop(
    columns=["NM_MUN", "NM_UF", "CD_UF"], errors="ignore"
)
basico = pd.read_csv(path + "ACM_basico_SP.csv", encoding="latin1", sep=";")

# Data Cleaning
columns_to_remove = [
    "CD_REGIAO", "NM_REGIAO", "CD_UF", "NM_UF", "CD_NU", "NM_NU",
    "CD_AGLOM", "NM_AGLOM", "CD_RGINT", "NM_RGINT", "CD_RGI", "NM_RGI",
    "CD_CONCURB", "NM_CONCURB", "AREA_KM2",
    "v0001", "v0002", "v0003", "v0004", "v0006",
]
basico = basico.drop(columns=columns_to_remove, errors="ignore")

# Data Standardization
for d in (basico, dom1, dom2, dom3):
    d["CD_MUN"] = d["CD_MUN"].astype(str)

# Data Merge (household characteristics only)
df = (
    basico.merge(dom1, on="CD_MUN", how="left")
    .merge(dom2, on="CD_MUN", how="left")
    .merge(dom3, on="CD_MUN", how="left")
)

print(f"Final consolidated dataset: {df.shape[0]} municipalities and {df.shape[1]} variables.")
df.to_csv("base_final_SP.csv", index=False, sep=";", encoding="utf-8")
print("base_final_SP.csv successfully generated!")

# 2. SUPPLEMENTARY VARIABLE (separate file, not used in the clustering)
df_favelas = pd.read_csv(path + "pop_favelas_m.csv", sep=";")
df_favelas["COD_MUNICIPIO"] = df_favelas["COD_MUNICIPIO"].astype(str)

# Keep all 645 municipalities; those without favelas receive 0
df_sup = df[["CD_MUN"]].merge(
    df_favelas[["COD_MUNICIPIO", "N_POP_DPPO"]],
    left_on="CD_MUN", right_on="COD_MUNICIPIO", how="left",
).drop(columns=["COD_MUNICIPIO"])
df_sup["N_POP_DPPO"] = df_sup["N_POP_DPPO"].fillna(0).astype(int)

df_sup.to_csv("favelas_suplementar_SP.csv", index=False, sep=";", encoding="utf-8")
print(f"favelas_suplementar_SP.csv generated: {df_sup.shape[0]} municipalities.")

# After running: upload base_final_SP.csv and favelas_suplementar_SP.csv to 1_DATA/processed/